# Trees and Ensembles

split loss를 직접 계산하고 작은 stump ensemble을 구현합니다.

핵심 셀은 Python 3 + NumPy만 사용합니다. 데이터 다운로드와 API 키가 필요하지 않습니다.

## 입력과 최적 stump

In [ ]:
import numpy as np
x=np.arange(1,9,dtype=float)
y=np.array([1,1,2,2,7,8,8,9.])
def stump_fit(x,y):
    best=None
    for t in (np.unique(x)[:-1]+np.unique(x)[1:])/2:
        left=x<=t
        a,b=y[left].mean(),y[~left].mean()
        pred=np.where(left,a,b); loss=np.mean((y-pred)**2)
        if best is None or loss<best[0]:best=(loss,t,a,b)
    return best
stump=stump_fit(x,y)
print(stump)

## Boosting 잔차

In [ ]:
fit=np.full_like(y,y.mean()); trace=[]
for m in range(20):
    loss,t,a,b=stump_fit(x,y-fit)
    fit += .2*np.where(x<=t,a,b)
    trace.append(np.mean((y-fit)**2))
print("first/last MSE", trace[0],trace[-1])
assert trace[-1]<trace[0]

## 범위 밖 예측

In [ ]:
_,t,a,b=stump
new=np.array([-10.,20.])
print("unseen inputs",new,"predictions",np.where(new<=t,a,b))

## 과제

- learning rate를 0.05와 1로 바꾸고 20round의 loss를 비교하세요.
- stump와 선형 least squares의 훈련 MSE를 비교하고 차이가 나는 입력 구조를 설명하세요.

## 해답과 검증

In [ ]:
for rate in [.05,1.]:
    f=np.full_like(y,y.mean())
    for _ in range(20):
        _,t,a,b=stump_fit(x,y-f); f+=rate*np.where(x<=t,a,b)
    print(rate,np.mean((y-f)**2))
A=np.c_[x,np.ones(len(x))]; w=np.linalg.lstsq(A,y,rcond=None)[0]
print("linear MSE",np.mean((A@w-y)**2),"stump MSE",stump[0])

## 선택: 실제 프레임워크

설치되어 있을 때만 실행합니다. 설치 명령이나 네트워크 접근은 자동으로 실행하지 않습니다.

In [ ]:
import importlib.util
if importlib.util.find_spec("sklearn"):
    from sklearn.ensemble import RandomForestRegressor
    model=RandomForestRegressor(n_estimators=50,max_depth=3,random_state=7)
    model.fit(x[:,None],y)
    print(model.predict(x[:,None]))
else:
    print("Optional sklearn example skipped; core stump and boosting completed")